In [0]:
from pyspark.sql import Row
from pyspark.sql.functions import to_timestamp

data = [
    (1, 101, "TV", 1000, "2024-01-01 10:00:00"),
    (2, 101, "TV", 1000, "2024-01-01 10:00:00"),   # exact duplicate
    (3, 101, "Mobile", 500, "2024-01-01 11:00:00"),
    (4, 102, "Shoes", 200, "2024-01-02 10:00:00"),
    (5, 102, "Shoes", 250, "2024-01-02 12:00:00"), # updated price
    (6, 103, "Laptop", 1500, "2024-01-03 09:00:00"),
    (7, 103, "Laptop", 1500, "2024-01-03 09:00:00"), # duplicate
    (8, 104, "Watch", 300, "2024-01-04 14:00:00")
]

columns = ["record_id","customer_id","product","amount","order_time"]

df = spark.createDataFrame(data, columns) \
          .withColumn("order_time", to_timestamp("order_time"))

df.show()

In [0]:
# 1. Remove identical rows.

df1 = df.dropDuplicates()

df1.show()

In [0]:
# 2. Keep only one record per customer.

df2 = df.dropDuplicates(["customer_id"])

df2.show()

In [0]:
# 3. Keep Latest Order Per Customer
# Use window + row_number. 

#df.show()

from pyspark.sql.window import Window
from pyspark.sql.functions import row_number,col

window_spec = Window.partitionBy("customer_id").orderBy("order_time")

df3 = df.withColumn("row_number",row_number().over(window_spec))

df3.filter(col("row_number") == 1).show()

In [0]:
# 4. Keep Highest Order Amount Per Customer

from pyspark.sql.functions import row_number, col
from pyspark.sql.window import Window

window_spec = Window.partitionBy("customer_id").orderBy(col("amount").desc())

df4 = df.withColumn("row_number", row_number().over(window_spec))

df4.filter(col("row_number") == 1).show()

In [0]:
# 5. Keep Earliest Order Per Customer

df.show()

from pyspark.sql.functions import min

df5  = df.groupBy("customer_id").agg(min("order_time"))

df5.show()


In [0]:
# 6. Deduplicate Based on (customer_id, product)

df.show()

df6 = df.dropDuplicates(["customer_id","product"])

df6.show()

In [0]:
# 7. Identify Duplicate Orders
#Find orders that appear more than once.

from pyspark.sql.functions import count

df7 = df.groupBy("customer_id", "product").agg(count("*").alias("count")).filter(col("count") > 1)

df7.show()

In [0]:
# 8. Keep First Order Per Customer

#df.show()

from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, col

window_spec = Window.partitionBy("customer_id").orderBy(col("order_time"))

df8 = df.withColumn("row_number", row_number().over(window_spec)).filter(col("row_number") == 1).drop("row_number").show()

In [0]:
# 10. Remove Duplicates After Join

products = spark.createDataFrame(
    [
        ("TV","Electronics"),
        ("Mobile","Electronics"),
        ("Shoes","Fashion"),
        ("Laptop","Electronics"),
        ("Watch","Accessories")
    ],
    ["product","category"]
)

joined = df.join(products,"product","left")

df10 = joined.dropDuplicates(["customer_id","product"])

df10.show()